# Auto-Improve Your Agent's Prompt with Simulation Feedback

Simulate diverse conversations to find where your agent fails, auto-optimize the prompt, and re-simulate to confirm the fix.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/future-agi/cookbooks/blob/cookbook/quickstart-notebooks/use-cases/simulation-optimization-loop.ipynb)

| Time | Difficulty |
|------|------------|
| 30 min | Intermediate |

You have a conversational agent with a minimal system prompt. It handles simple questions fine, but falls apart on edge cases: it misses urgent escalations, hallucinates tool commands, drops context mid-conversation, and responds to frustrated users with the same calm tutorial tone. You know these failures exist because users complain, but you don't know how widespread they are or how to systematically fix them.

This cookbook walks you through a closed loop: simulate 20 diverse conversations to surface failures, auto-optimize the prompt based on what went wrong, then re-simulate to verify the fix actually worked.

**Prerequisites:**
- FutureAGI account: [app.futureagi.com](https://app.futureagi.com)
- API keys: `FI_API_KEY` and `FI_SECRET_KEY` (see [Get your API keys](https://docs.futureagi.com/docs/admin-settings))
- OpenAI API key (`OPENAI_API_KEY`)
- Python 3.9+

## Install

In [ ]:
!pip install ai-evaluation agent-simulate agent-opt openai

In [ ]:
import os

os.environ["FI_API_KEY"] = "your-fi-api-key"
os.environ["FI_SECRET_KEY"] = "your-fi-secret-key"
os.environ["OPENAI_API_KEY"] = "your-openai-key"

## Step 1: Define the agent you want to improve

Start with the agent you're trying to fix. This example uses a helpdesk-style agent with three tools (status checks, documentation lookup, and engineering escalation), but the same loop works for any conversational agent. The system prompt is deliberately minimal so we can measure the gap.

In [ ]:
import os
import json
from openai import AsyncOpenAI

client = AsyncOpenAI()

SYSTEM_PROMPT = """You are a technical support agent for a cloud infrastructure platform. Help developers with their issues."""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "check_service_status",
            "description": "Check the current status of a platform service (compute, networking, storage, dns, ssl, billing)",
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {"type": "string", "description": "Service name to check (compute, networking, storage, dns, ssl, billing)"}
                },
                "required": ["service"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "lookup_docs",
            "description": "Search platform documentation for troubleshooting steps, CLI commands, or configuration guides",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The topic or error message to look up"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineering",
            "description": "Escalate a critical issue to the on-call engineering team",
            "parameters": {
                "type": "object",
                "properties": {
                    "severity": {"type": "string", "description": "Issue severity: P0 (production down), P1 (degraded), P2 (non-critical)"},
                    "summary": {"type": "string", "description": "Brief summary of the issue for the on-call engineer"},
                    "affected_service": {"type": "string", "description": "Which service is affected"}
                },
                "required": ["severity", "summary", "affected_service"]
            }
        }
    }
]


# Mock tool implementations
def check_service_status(service: str) -> dict:
    statuses = {
        "compute": {"status": "operational", "uptime": "99.98%", "last_incident": "2025-02-14"},
        "networking": {"status": "degraded", "issue": "Elevated latency in us-east-1", "since": "2025-03-10T14:30Z"},
        "storage": {"status": "operational", "uptime": "99.99%", "last_incident": "2025-01-22"},
        "dns": {"status": "operational", "uptime": "99.97%", "last_incident": "2025-03-01"},
        "ssl": {"status": "operational", "uptime": "99.99%", "last_incident": "2025-02-28"},
        "billing": {"status": "operational", "uptime": "100%", "last_incident": "N/A"},
    }
    return statuses.get(service.lower(), {"error": f"Unknown service: {service}"})

def lookup_docs(query: str) -> dict:
    return {
        "title": "Platform Troubleshooting Guide",
        "content": "Common deployment issues: 1) Check your config syntax with `app validate`. "
                   "2) Verify environment variables in project settings. "
                   "3) Review build logs at Dashboard > Deployments > select deployment > Logs tab. "
                   "4) For DNS propagation, allow up to 48 hours after domain configuration. "
                   "5) SSL certificates auto-renew 30 days before expiry.",
        "cli_commands": {
            "deploy": "app deploy --project <name> --env production",
            "logs": "app logs <deployment-id> --tail 100",
            "status": "app status --project <name>",
            "rollback": "app rollback <deployment-id> --to <previous-id>",
            "validate": "app validate ./config.yml",
        },
    }

def escalate_to_engineering(severity: str, summary: str, affected_service: str) -> dict:
    return {
        "status": "escalated",
        "ticket_id": "INC-2025-0847",
        "assigned_to": "On-call SRE",
        "sla": {"P0": "15 minutes", "P1": "1 hour", "P2": "4 hours"}.get(severity, "4 hours"),
    }


async def handle_message(messages: list) -> str:
    """Send messages to OpenAI and handle tool calls."""
    response = await client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )

    msg = response.choices[0].message

    if msg.tool_calls:
        messages.append(msg)
        for tool_call in msg.tool_calls:
            fn_name = tool_call.function.name
            fn_args = json.loads(tool_call.function.arguments)

            tool_fn = {
                "check_service_status": check_service_status,
                "lookup_docs": lookup_docs,
                "escalate_to_engineering": escalate_to_engineering,
            }
            result = tool_fn.get(fn_name, lambda **_: {"error": "Unknown tool"})(**fn_args)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result),
            })

        followup = await client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOLS,
        )
        return followup.choices[0].message.content

    return msg.content

That one-line system prompt covers the happy path. But there's nothing about severity classification, when to escalate, which CLI commands are valid, or how to handle a developer whose production is down. The model will improvise, and improvisation during an outage is how you lose users.

## Step 2: Generate scenarios and run the baseline simulation

Before you can find failures, you need diverse conversations that cover the range of real-world interactions: routine questions, urgent incidents, frustrated users, and edge cases.

**Register your agent in the dashboard:**

1. Go to **Simulate** > **Agent Definition** > **Create agent definition**
2. Fill in:

| Step | Field | Value |
|---|---|---|
| Basic Info | **Agent type** | `Chat` |
| Basic Info | **Agent name** | `my-support-agent` |
| Basic Info | **Select language** | `English` |
| Configuration | **Model Used** | `gpt-4o-mini` |
| Behaviour | **Prompt / Chains** | *(paste the system prompt from Step 1)* |
| Behaviour | **Commit Message** | `v1: minimal prompt, no escalation rules` |

3. Click **Create**

**Generate scenarios:**

1. Go to **Simulate** > **Scenarios** > **Create New Scenario**
2. Select **Workflow builder**
3. Fill in:

| Field | Value |
|---|---|
| **Scenario Name** | `stress-test-v1` |
| **Description** | Developers troubleshooting deployment failures, DNS propagation delays, SSL certificate errors, networking outages, billing disputes, and production-down emergencies. Mix of routine questions and high-severity incidents. |
| **Choose source** | `my-support-agent` (Agent Definition) |
| **Choose version** | `v1` |
| **No. of scenarios** | `20` |

4. Click **Create**

The platform generates 20 realistic scenarios based on your agent definition. Each gets an automatically assigned persona (patient, frustrated, confused, technical, impatient, and others), so you get diverse conversation styles without manual setup.

**Configure and run the simulation:**

1. Go to **Simulate** > **Run Simulation** > **Create a Simulation**
2. Fill in:

| Step | Field | Value |
|---|---|---|
| Details | **Simulation name** | `baseline-v1` |
| Details | **Choose Agent definition** | `my-support-agent` |
| Details | **Choose version** | `v1` |
| Scenarios | **Select scenario** | `stress-test-v1` |
| Evaluations | **Add Evaluations** | Select **Conversational agent evaluation** group |

3. Click **Run Simulation**

The Conversational agent evaluation group runs 10 metrics automatically: context retention, query handling, loop detection, escalation handling, prompt conformance, and more.

Now connect your agent to the simulation via the SDK.

> **Warning:** The `run_test_name` must exactly match the simulation name you entered in the dashboard (e.g., `baseline-v1`). A mismatch returns a 404.

In [ ]:
import os
from fi.simulate import TestRunner, AgentInput

runner = TestRunner(
    api_key=os.environ["FI_API_KEY"],
    secret_key=os.environ["FI_SECRET_KEY"],
)


async def agent_callback(input: AgentInput) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for msg in input.messages:
        messages.append(msg)

    return await handle_message(messages)


report = await runner.run_test(
    run_test_name="baseline-v1",
    agent_callback=agent_callback,
)
print(f"Simulation complete: {len(report.results)} conversations processed")

## Step 3: Find where your agent fails

Once the simulation finishes, open **Simulate** > click `baseline-v1` > go to the **Analytics** tab.

You'll see aggregate scores across all 20 conversations for each evaluation metric. With a minimal prompt, expect a split: routine questions score well, but the lower-scoring conversations are where it gets interesting. Switch to the **Chat Details** tab and click into them to see full transcripts with per-message eval annotations.

Common failure patterns you'll see with a minimal prompt:

- **Missed escalations.** A developer says "our production site has been down for 45 minutes" and the agent walks them through generic troubleshooting instead of immediately escalating.
- **Hallucinated commands.** The agent suggests CLI commands that don't exist (`app restart --force`, `app config set dns.ttl 300`). The actual commands are `app deploy`, `app logs`, `app status`, `app rollback`, and `app validate`.
- **Ignoring known incidents.** Networking is degraded in us-east-1, but the agent doesn't check service status before telling the developer to debug their own configuration.
- **Wrong tone for the situation.** A developer whose production is down gets the same measured, tutorial-style response as someone asking a casual question.
- **Dropped context.** A developer shares their project name and error message, then the agent asks for that same information two messages later.

You can also spot-check specific conversations from the SDK:

In [ ]:
import os
import json
from fi.evals import Evaluator

evaluator = Evaluator(
    fi_api_key=os.environ["FI_API_KEY"],
    fi_secret_key=os.environ["FI_SECRET_KEY"],
)

# Paste a conversation from the Chat Details tab
conversation = [
    {"role": "user", "content": "Our production app has been returning 502s for 30 minutes. We're losing customers. This is critical."},
    {"role": "assistant", "content": "I'd be happy to help! Let me walk you through some troubleshooting steps. First, can you check your deployment logs?"},
    {"role": "user", "content": "I already checked the logs. There's nothing useful. This is a P0. Can you escalate this NOW?"},
    {"role": "assistant", "content": "I understand your concern. Have you tried redeploying your application? You can use `app redeploy --force` to force a fresh deployment."},
]

for metric in ["customer_agent_human_escalation", "customer_agent_query_handling", "customer_agent_context_retention"]:
    result = evaluator.evaluate(
        eval_templates=metric,
        inputs={"conversation": json.dumps(conversation)},
        model_name="turing_small",
    )
    eval_result = result.eval_results[0]
    score = eval_result.output[0] if isinstance(eval_result.output, list) else eval_result.output
    print(f"{metric}: {score}")
    print(f"  Reason: {eval_result.reason}\n")

The eval reasons tell you exactly what went wrong: which escalation was missed, which command was fabricated, which context was lost. These reasons become the input for the next step.

See [Conversation Eval](https://docs.futureagi.com/docs/cookbook/quickstart/conversation-eval) for all 10 metrics in the evaluation group.

## Step 4: Get actionable fix recommendations

Reading 20 transcripts tells you what's wrong. The Fix My Agent feature tells you what to do about it.

1. Go to **Simulate** > click `baseline-v1`
2. Click the **Fix My Agent** button (top-right)

The diagnostic drawer opens with two categories:

**Fixable Recommendations** are prompt-level changes you can apply directly:
- **Agent Level**: broad improvements like "add severity classification framework" or "restrict CLI commands to the documented set"
- **Branch Level**: domain-specific issues grouped by topic, each linked to the specific conversations where the failure occurred

**Non-Fixable Recommendations** are infrastructure-level issues that need code changes, like "agent lacks access to real-time incident status" or "no mechanism to pull deployment history."

Instead of manually rewriting the prompt based on these insights, you can feed them directly into the optimizer.

> **Tip:** Fix My Agent works best with at least **15 completed conversations**. If you ran fewer, increase the scenario count and re-run before using this feature.

## Step 5: Auto-optimize the prompt

You have two paths: one-click in the dashboard, or programmatic via the SDK.

**Dashboard (one-click):**

Inside the Fix My Agent drawer:

1. Click **Optimize My Agent**
2. Enter a name for the optimization run (e.g., `v1-optimization`)
3. Select **MetaPrompt** as the optimizer
4. Select a language model (e.g., `gpt-4o`)
5. Click **Start Optimizing your agent**

Results appear in the **Optimization Runs** tab once complete.

**SDK (programmatic):**

Use the SDK when you want more control: custom training examples, specific evaluation metrics, or CI/CD integration.

In [ ]:
import os
from fi.opt.generators import LiteLLMGenerator
from fi.opt.optimizers import MetaPromptOptimizer
from fi.opt.base import Evaluator
from fi.opt.datamappers import BasicDataMapper

# Build a dataset from the failing conversations you identified.
# Each example: the user's message and the ideal agent response.
dataset = [
    {
        "input": "Our production app has been returning 502s for 30 minutes. Losing revenue every minute. Fix this NOW.",
        "output": "I can see this is critical. Let me check the service status immediately. I'm seeing elevated latency in us-east-1 networking, which is likely related. I'm escalating this as a P0 right now to the on-call SRE team. They'll respond within 15 minutes per our SLA. Ticket INC-2025-0847 has been created. In the meantime, if you have a rollback candidate, you can run `app rollback <deployment-id> --to <previous-id>` to revert to the last working deployment.",
    },
    {
        "input": "How do I set up a custom domain with SSL? I just deployed my first app.",
        "output": "Here's the process: 1) Go to Dashboard > Domains > Add Domain and enter your custom domain. 2) Add the DNS records (CNAME or A) provided by the platform to your domain registrar. 3) SSL certificates are provisioned automatically once DNS propagates. Allow up to 48 hours, though it's usually faster. You can verify with `app status --project <name>`.",
    },
    {
        "input": "My deployment keeps failing with exit code 137. I've tried `app restart --force` but nothing works.",
        "output": "Exit code 137 means the process was killed, usually an out-of-memory (OOM) issue. Note that `app restart --force` isn't a valid command. The available commands are `app deploy`, `app logs`, `app status`, `app rollback`, and `app validate`. To debug this: run `app logs <deployment-id> --tail 100` to check for memory spikes, then review your config memory limits with `app validate ./config.yml`.",
    },
    {
        "input": "I'm getting charged for resources I deleted two weeks ago. This is unacceptable. I want a refund.",
        "output": "I understand your frustration. Let me check the billing service status first. The billing system is operational, so this may be a resource cleanup issue: some resources like load balancers persist after project deletion unless explicitly removed. Billing disputes and refund requests need account-level access, so let me escalate this to the billing team. What's the best email to reach you for the follow-up?",
    },
    {
        "input": "DNS isn't resolving for my domain. I set it up 3 days ago. What's going on?",
        "output": "Three days is past the typical propagation window, so something else is likely going on. Let me check DNS service status. DNS is operational, so the issue is probably in your configuration. Verify that the CNAME or A records you added match what's shown in Dashboard > Domains. Also check for conflicting records (like an existing A record overriding the CNAME). Can you share your domain name so I can look at the specific configuration?",
    },
]

# Teacher model rewrites the prompt
teacher = LiteLLMGenerator(model="gpt-4o", prompt_template="{prompt}")

optimizer = MetaPromptOptimizer(teacher_generator=teacher)

evaluator = Evaluator(
    eval_template="completeness",
    eval_model_name="turing_small",
)

data_mapper = BasicDataMapper(
    key_map={
        "input": "input",
        "output": "generated_output",
    }
)

result = optimizer.optimize(
    evaluator=evaluator,
    data_mapper=data_mapper,
    dataset=dataset,
    initial_prompts=[SYSTEM_PROMPT],
    task_description="Improve a support agent prompt. The agent should classify incident severity, escalate P0 outages immediately, only suggest documented CLI commands, handle frustrated users with appropriate urgency, and check service status before troubleshooting. It has three tools: check_service_status, lookup_docs, and escalate_to_engineering.",
    num_rounds=5,
    eval_subset_size=5,
)

print(f"Optimization complete")
print(f"Best score: {result.final_score:.3f}")
print(f"\nOptimized prompt:")
print("-" * 60)
best_prompt = result.best_generator.get_prompt_template()
print(best_prompt)
print("-" * 60)

# Show round-by-round progress
print("\nOptimization history:")
for i, iteration in enumerate(result.history):
    print(f"  Round {i+1}: score={iteration.average_score:.3f}")

The optimizer iterates through multiple rounds. Each round, the teacher model analyzes which examples the current prompt handles poorly, hypothesizes why, and rewrites the prompt to address the gaps. After 5 rounds, you get the best-performing variant.

See [Prompt Optimization](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-optimization) for the full SDK walkthrough. To compare MetaPrompt against other strategies, see [Compare Optimizers](https://docs.futureagi.com/docs/cookbook/quickstart/compare-optimizers).

## Step 6: Re-simulate with the optimized prompt

The optimizer gives you a better prompt, but "better on 5 examples" and "better on 20 diverse conversations" are different claims. Re-simulation is how you verify.

Take the optimized prompt from the previous step. A typical result includes severity classification rules, escalation protocols, a whitelist of valid CLI commands, urgency-aware tone guidance, and tool usage instructions. Use whatever your optimizer produced.

**Update in the dashboard:**

1. Go to **Simulate** > **Agent Definition** > open `my-support-agent`
2. Click **Create new version**
3. Paste the optimized prompt, set commit message to `v2: optimized with severity classification, escalation rules, CLI guardrails`
4. Click **Create**

**Create the v2 simulation:**

| Field | Value |
|---|---|
| **Simulation name** | `optimized-v2` |
| **Agent definition** | `my-support-agent` |
| **Version** | `v2` |
| **Scenario** | Create new with 20 scenarios from v2, or reuse `stress-test-v1` |
| **Evaluations** | **Conversational agent evaluation** group |

**Run the simulation with the updated prompt:**

In [ ]:
import os
from fi.simulate import TestRunner, AgentInput

runner = TestRunner(
    api_key=os.environ["FI_API_KEY"],
    secret_key=os.environ["FI_SECRET_KEY"],
)

# Assign the optimized prompt from Step 5:
OPTIMIZED_PROMPT = result.best_generator.get_prompt_template()


async def agent_callback(input: AgentInput) -> str:
    messages = [{"role": "system", "content": OPTIMIZED_PROMPT}]
    for msg in input.messages:
        messages.append(msg)

    return await handle_message(messages)


report = await runner.run_test(
    run_test_name="optimized-v2",
    agent_callback=agent_callback,
)
print(f"Simulation complete: {len(report.results)} conversations processed")

Open the Analytics tab and compare v2 results against v1. The same types of personas (frustrated, impatient, confused), but now your agent has explicit instructions. Look for improvements in the specific areas that were failing:

- Urgent incidents should trigger immediate escalation, not generic troubleshooting
- CLI command references should only include valid commands
- The agent should check service status before telling users to debug their own config
- Frustrated users should get urgency-matched responses
- Context (project names, deployment IDs, error messages) should persist across the conversation

Click into the Chat Details tab and read a few conversations side by side with v1 transcripts. The qualitative difference in how the agent handles urgency, uses tools proactively, and knows when to stop troubleshooting and escalate is often more telling than aggregate scores.

## Step 7: Repeat the loop as your agent evolves

You just completed one cycle:

```
Simulate > Find failures > Optimize prompt > Re-simulate > Confirm fix
```

This loop applies every time your agent changes:

- **You add a new tool.** Simulation reveals the agent doesn't know when to use it. Run the loop.
- **Your product adds a new feature.** Users start asking about it and the agent has no instructions. Simulation catches the gap. Run the loop.
- **You change the underlying model.** Behaviors shift. Simulation quantifies the difference. Run the loop.

Each iteration tightens the feedback. The first simulation shows everything that's broken. Optimization fixes the worst failures. Re-simulation catches what's left. Over time, you're building a prompt that's been pressure-tested against the full range of user behavior, not just the handful of test cases you thought of manually.

> **Tip:** For a more rigorous before/after comparison, use the Experimentation feature to run the same dataset against both prompts with weighted metric scoring. See [Experimentation](https://docs.futureagi.com/docs/cookbook/quickstart/experimentation-compare-prompts) for details.

## What you solved

You built a closed-loop improvement workflow: simulation discovers failures at scale, optimization fixes the prompt automatically, and re-simulation confirms the fix.

- **Failures you couldn't see manually** surfaced by simulating 20 diverse conversations with varied personas
- **Vague "the prompt needs work" feeling** replaced with specific, metric-backed failure patterns and Fix My Agent recommendations
- **Hours of manual prompt rewriting** replaced with auto-optimization that targets the exact gaps simulation found
- **"Did the fix actually work?" uncertainty** eliminated by re-simulating with the same conversation types and comparing scores

**Explore further:**
- [Chat Simulation](https://docs.futureagi.com/docs/cookbook/quickstart/chat-simulation-personas): Custom personas, scenario builders, tool-calling evaluation
- [Prompt Optimization](https://docs.futureagi.com/docs/cookbook/quickstart/prompt-optimization): MetaPrompt, Bayesian Search, and the full SDK workflow
- [Compare Optimizers](https://docs.futureagi.com/docs/cookbook/quickstart/compare-optimizers): ProTeGi, GEPA, PromptWizard: pick the right strategy
- [Conversation Eval](https://docs.futureagi.com/docs/cookbook/quickstart/conversation-eval): All 10 metrics in the Conversational agent evaluation group